# 05 — DG38 Results Aggregation, Three-Seed Statistics (`ddof=1`) & Completion Report

**Chức năng của Notebook này:**
1. Quét toàn bộ thư mục `runs/` để cập nhật `experiment_registry.json`.
2. Tính toán thống kê **Mean ± Sample Std (`ddof=1`)** cho từng nhóm thí nghiệm 3 seeds (`42`, `123`, `2026`).
   - Tự động kiểm tra tính đồng nhất (cùng mô hình, cùng manifest SHA-256, cùng giao thức huấn luyện).
   - Nếu nhóm nào mới có 1 hoặc 2 seeds thì tự động gắn nhãn `INCOMPLETE` (không giả mạo độ lệch chuẩn của 3 seeds).
3. Xuất bảng đối chiếu kết quả mới với kết quả gốc trong báo cáo và cập nhật trạng thái hoàn thành DG38 (`dg38_completion_status.md`, `cu_budget_report.md`).

In [ ]:
# [CELL 1] Quét toàn bộ kết quả thực nghiệm & hiển thị bảng trạng thái ngân sách CU
import os
import sys
import json
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")

DRIVE_ROOT = Path(os.environ.get("DG38_DRIVE_ROOT", "/content/drive/MyDrive/DG38_Colab_Package"))
REPO_ROOT = DRIVE_ROOT / "code" if (DRIVE_ROOT / "code" / "model.py").exists() else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

EXP_DIR = REPO_ROOT / "evaluation_runs" / "dg38_additional_experiments"
ARTIFACTS_DIR = (DRIVE_ROOT / "artifacts") if DRIVE_ROOT.exists() else EXP_DIR

from evaluation_runs.dg38_additional_experiments.dg38_stats_and_analysis import (
    scan_and_update_registry,
    aggregate_seed_group,
)
from evaluation_runs.dg38_additional_experiments.cu_budget_tracker import CUBudgetTracker

registry = scan_and_update_registry(ARTIFACTS_DIR)
print(f"Tổng số thực nghiệm đã ghi nhận trong registry: {len(registry.get('experiments', {}))}")

tracker = CUBudgetTracker(EXP_DIR / "cu_budget_ledger.json")
print(tracker.format_budget_table_markdown())

In [ ]:
# [CELL 2] Phân nhóm theo kiến trúc và kiểm tra trạng thái 3-Seed (42, 123, 2026)
experiments = list(registry.get("experiments", {}).values())

groups = {}
for exp in experiments:
    task = exp.get("task", "unknown")
    model = exp.get("model_name", "unknown")
    proto = exp.get("protocol_id", "default")
    gid = f"{task}::{model}::{proto}"
    groups.setdefault(gid, []).append(exp)

for gid, runs in sorted(groups.items()):
    task = runs[0].get("task")
    metric_keys = (
        ["val_macro_f1", "test_accuracy", "test_macro_f1"]
        if task == "classification"
        else ["val_clean_mAP_50", "val_clean_mAP_50_95", "val_clean_f1"]
    )
    try:
        agg = aggregate_seed_group(gid, runs, metric_keys=metric_keys)
        print(f"\n=== NHÓM: {gid} | Trạng thái: {agg['overall_status']} ({agg['n_completed_seeds']}/3 seeds) ===")
        for mk, mstat in agg["metrics_summary"].items():
            print(f"  - {mk:22s}: {mstat['formatted']}")
    except ValueError as err:
        print(f"\n[LỖI ĐỒNG NHẤT NHÓM {gid}]: {err}")